## Project Description: Next Word Prediction Using LSTM
#### Project Overview:

This project aims to develop a deep learning model for predicting the next word in a given sequence of words. The model is built using Long Short-Term Memory (LSTM) networks, which are well-suited for sequence prediction tasks. The project includes the following steps:

1- Data Collection: We use the text of Shakespeare's "Hamlet" as our dataset. This rich, complex text provides a good challenge for our model.

2- Data Preprocessing: The text data is tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into training and testing sets.

3- Model Building: An LSTM model is constructed with an embedding layer, two LSTM layers, and a dense output layer with a softmax activation function to predict the probability of the next word.

4- Model Training: The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when the loss stops improving.

5- Model Evaluation: The model is evaluated using a set of example sentences to test its ability to predict the next word accurately.

6- Deployment: A Streamlit web application is developed to allow users to input a sequence of words and get the predicted next word in real-time.

In [1]:
# Data collection
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\jenis\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [2]:
# load the dataset
data = gutenberg.raw('shakespeare-hamlet.txt')

In [3]:
# savle dataset to a file
with open('hamlet.txt', 'w') as file:
    file.write(data)

In [4]:
# Data preprocessing
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [5]:
with open('hamlet.txt', 'r') as file:
    text = file.read().lower()
 
text = re.sub(r'[^a-zA-Z\s]', '', text)
 
# Tokenize
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
 
print(f"Total unique words: {total_words}")

Total unique words: 4798


In [6]:
MAX_LEN_CAP = 20

In [7]:
# Create input sequences
input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
 
    if len(token_list) < 2:
        continue
 
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[max(0, i - MAX_LEN_CAP): i + 1]
        input_sequences.append(n_gram_sequence)

In [8]:
input_sequences

[[2, 684],
 [2, 684, 5],
 [2, 684, 5, 46],
 [2, 684, 5, 46, 42],
 [2, 684, 5, 46, 42, 1857],
 [2, 684, 5, 46, 42, 1857, 1858],
 [1168, 1859],
 [1168, 1859, 1860],
 [1168, 1859, 1860, 1861],
 [58, 407],
 [58, 407, 3],
 [58, 407, 3, 1169],
 [58, 407, 3, 1169, 181],
 [58, 407, 3, 1169, 181, 1862],
 [407, 1170],
 [407, 1170, 64],
 [408, 162],
 [408, 162, 375],
 [408, 162, 375, 22],
 [408, 162, 375, 22, 248],
 [408, 162, 375, 22, 248, 877],
 [19, 69],
 [450, 224],
 [450, 224, 258],
 [450, 224, 258, 2],
 [450, 224, 258, 2, 31],
 [408, 407],
 [450, 26],
 [408, 7],
 [408, 7, 44],
 [408, 7, 44, 62],
 [408, 7, 44, 62, 1863],
 [408, 7, 44, 62, 1863, 97],
 [408, 7, 44, 62, 1863, 97, 19],
 [408, 7, 44, 62, 1863, 97, 19, 567],
 [450, 66],
 [450, 66, 52],
 [450, 66, 52, 1864],
 [450, 66, 52, 1864, 568],
 [450, 66, 52, 1864, 568, 376],
 [450, 66, 52, 1864, 568, 376, 81],
 [450, 66, 52, 1864, 568, 376, 81, 4],
 [450, 66, 52, 1864, 568, 376, 81, 4, 322],
 [450, 66, 52, 1864, 568, 376, 81, 4, 322, 1169],

In [9]:
# Pad sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
)

In [10]:
# Create predictors and labels
X, y = input_sequences[:, :-1], input_sequences[:, -1]

In [11]:
y = y.astype(np.int32)

In [12]:
y

array([ 684,    5,   46, ..., 1044,    5,  193], dtype=int32)

In [13]:
# split data into train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")
print(f"Max sequence len : {max_sequence_len}")

Training samples : 20489
Testing samples  : 5123
Max sequence len : 14


In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping  # single import (FIX 6)
 
model = Sequential([
    Embedding(total_words, 100, input_length=max_sequence_len - 1),
    LSTM(150, return_sequences=True),
    Dropout(0.2),
    LSTM(100),
    Dropout(0.2),
    Dense(total_words, activation='softmax')
])
 
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
 
model.summary()

c:\Users\jenis\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [17]:
# train model
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=128,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
161/161 ━━━━━━━━━━━━━━━━━━━━ 17s 79ms/step - accuracy: 0.0312 - loss: 7.0145 - val_accuracy: 0.0340 - val_loss: 6.7827
Epoch 2/50
161/161 ━━━━━━━━━━━━━━━━━━━━ 12s 73ms/step - accuracy: 0.0348 - loss: 6.6081 - val_accuracy: 0.0340 - val_loss: 6.8173
Epoch 3/50
161/161 ━━━━━━━━━━━━━━━━━━━━ 12s 73ms/step - accuracy: 0.0348 - loss: 6.4731 - val_accuracy: 0.0340 - val_loss: 6.8296
Epoch 4/50
161/161 ━━━━━━━━━━━━━━━━━━━━ 12s 74ms/step - accuracy: 0.0366 - loss: 6.3991 - val_accuracy: 0.0418 - val_loss: 6.8743
